# Federated Baselines: Local, Pooled, FedAvg, FedProx


In [1]:
import os
import sys
import copy
from pathlib import Path

root_dir = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import pandas as pd
import torch
from src.seed import seed_everything
from src.data import load_all_sites, load_pooled_data
from src.models import Baseline, get_loss_fn, evaluate_model
from src.local import run_local_only, run_pooled
from src.fl import run_federated

config = {
    "dataset": "fed_heart_disease",
    "seeds": list(range(42, 52)),
    "epochs": 50,
    "rounds": 50,
    "local_epochs": 1,
    "batch_size": 4,
    "lr": 0.001,
    "mu_fedprox": 0.01,
    "output_csv": str(root_dir / "results" / "e1.csv"),
}

Verification of mathematical sanity checks: single-client FedAvg parity and identical-clients FedAvg parity.

In [2]:
seed_everything(42)
x_test = torch.randn(20, 13)
y_test = torch.randint(0, 2, (20, 1)).float()

# Check 1: 1-client FedAvg == Local training
seed_everything(100)
m_loc = Baseline()
opt_loc = torch.optim.SGD(m_loc.parameters(), lr=0.01)
loss_fn = get_loss_fn()
for _ in range(10):
    for i in range(0, len(x_test), 4):
        xb, yb = x_test[i:i + 4], y_test[i:i + 4]
        opt_loc.zero_grad()
        loss_fn(m_loc(xb), yb).backward()
        opt_loc.step()

seed_everything(100)
m_fed = Baseline()
for _ in range(2):
    m_c = copy.deepcopy(m_fed)
    opt_c = torch.optim.SGD(m_c.parameters(), lr=0.01)
    for _ in range(5):
        for i in range(0, len(x_test), 4):
            xb, yb = x_test[i:i + 4], y_test[i:i + 4]
            opt_c.zero_grad()
            loss_fn(m_c(xb), yb).backward()
            opt_c.step()
    m_fed.load_state_dict(m_c.state_dict())

diff1 = sum(torch.norm(p1 - p2).item() for p1, p2 in zip(m_loc.parameters(), m_fed.parameters()))
print(f"Check 1 (1-client FedAvg == Local): diff = {diff1:.10f}")

# Check 2: Identical clients FedAvg == Local training
seed_everything(100)
m_fed2 = Baseline()
for _ in range(2):
    c_weights = []
    for _ in range(3):
        m_c = copy.deepcopy(m_fed2)
        opt_c = torch.optim.SGD(m_c.parameters(), lr=0.01)
        for _ in range(5):
            for i in range(0, len(x_test), 4):
                xb, yb = x_test[i:i + 4], y_test[i:i + 4]
                opt_c.zero_grad()
                loss_fn(m_c(xb), yb).backward()
                opt_c.step()
        c_weights.append(m_c.state_dict())
    avg_sd = {k: torch.stack([w[k] for w in c_weights]).mean(dim=0) for k in c_weights[0]}
    m_fed2.load_state_dict(avg_sd)

diff2 = sum(torch.norm(p1 - p2).item() for p1, p2 in zip(m_loc.parameters(), m_fed2.parameters()))
print(f"Check 2 (Identical clients FedAvg == Local): diff = {diff2:.10f}")

Check 1 (1-client FedAvg == Local): diff = 0.0000000000
Check 2 (Identical clients FedAvg == Local): diff = 0.0000000186


Loading hospital datasets and pooled data splits.

In [3]:
sites_data = load_all_sites()
pooled_tr = load_pooled_data(train=True)
pooled_te = load_pooled_data(train=False)
print("Loaded 4 sites and pooled dataset splits.")

Loaded 4 sites and pooled dataset splits.


Execution of local-only, pooled, FedAvg, and FedProx across 10 random seeds.

In [4]:
records = []
print("Running local-only baselines...")
records.extend(run_local_only(sites_data, config["seeds"], epochs=config["epochs"], lr=config["lr"], batch_size=config["batch_size"]))

print("Running pooled baselines...")
records.extend(run_pooled(sites_data, pooled_tr, pooled_te, config["seeds"], epochs=config["epochs"], lr=config["lr"], batch_size=config["batch_size"]))

print("Running FedAvg...")
records.extend(run_federated(sites_data, pooled_te, config["seeds"], rounds=config["rounds"], local_epochs=config["local_epochs"], lr=config["lr"], batch_size=config["batch_size"], mu=0.0, method="fedavg"))

print("Running FedProx...")
records.extend(run_federated(sites_data, pooled_te, config["seeds"], rounds=config["rounds"], local_epochs=config["local_epochs"], lr=config["lr"], batch_size=config["batch_size"], mu=config["mu_fedprox"], method="fedprox"))

df_results = pd.DataFrame(records)
os.makedirs(os.path.dirname(config["output_csv"]), exist_ok=True)
df_results.to_csv(config["output_csv"], index=False)
print(f"Saved {len(df_results)} rows to {config['output_csv']}")

Running local-only baselines...


Running pooled baselines...


Running FedAvg...


Running FedProx...


Saved 9500 rows to C:\Users\Soumadipta Konar\Desktop\fl-hospital-consortium\results\e1.csv


Summary table reporting final round accuracy mean and standard deviation across 10 seeds per hospital and overall.

In [5]:
final_df = df_results[df_results["round"] == config["rounds"]]
summary = final_df.groupby(["method", "site"])["metric_value"].agg(["mean", "std"]).reset_index()
summary["formatted"] = summary.apply(lambda r: f"{r['mean']:.3f} +/- {r['std']:.3f}", axis=1)
pivot_table = summary.pivot(index="site", columns="method", values="formatted")
print(pivot_table.to_string())

method           fedavg          fedprox       local-only           pooled
site                                                                      
0       0.773 +/- 0.012  0.773 +/- 0.012  0.750 +/- 0.012  0.777 +/- 0.012
1       0.778 +/- 0.016  0.778 +/- 0.016  0.752 +/- 0.011  0.765 +/- 0.004
2       0.675 +/- 0.049  0.675 +/- 0.049  0.775 +/- 0.148  0.750 +/- 0.000
3       0.591 +/- 0.024  0.591 +/- 0.024  0.700 +/- 0.041  0.649 +/- 0.009
all     0.782 +/- 0.010  0.782 +/- 0.010              NaN  0.796 +/- 0.003
